# 02 - Data Quality dan Profiling Lanjut

Notebook ini melanjutkan hasil `01_data_profiling.ipynb`. Tahap awal mencakup pembacaan data, profiling DuckDB `SUMMARIZE`, serta deteksi missing values pada dataset berita `idn-news-az`.

## Persiapan data

In [1]:
from pathlib import Path

import duckdb
import polars as pl

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Root project tidak ditemukan.")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "idn-news-az" / "data_files"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"
PARQUET_FILES = sorted(RAW_DIR.glob("*.parquet"))
if not PARQUET_FILES:
    raise FileNotFoundError(f"Tidak ada file Parquet di {RAW_DIR}")

RAW_GLOB = str(RAW_DIR / "*.parquet")
print(f"Polars {pl.__version__} | DuckDB {duckdb.__version__}")
print(f"Jumlah file: {len(PARQUET_FILES):,}")

Polars 1.44.2 | DuckDB 1.5.6
Jumlah file: 441


In [2]:
raw_lf = pl.scan_parquet(RAW_GLOB)
schema = raw_lf.collect_schema()
required_columns = ["date", "link", "title", "text"]
missing_columns = set(required_columns) - set(schema.names())
if missing_columns:
    raise KeyError(f"Kolom wajib tidak ditemukan: {sorted(missing_columns)}")

jumlah_baris = raw_lf.select(pl.len()).collect().item()
print(f"Ukuran awal: {jumlah_baris:,} baris x {len(schema)} kolom")
raw_lf.head(5).collect()

Ukuran awal: 1,149,789 baris x 4 kolom


date,link,title,text
str,str,str,str
"""2019-05-10""","""https://www.cnbcindonesia.com/…","""Potret Jemaah Haji Mulai Melak…","""Haji tahun ini merupakan tanta…"
"""2019-08-19""","""https://www.cnbcindonesia.com/…","""Cek Warga China Mencoba Robot …","""Seorang anggota staf mengambil…"
"""2019-08-26""","""https://www.cnbcindonesia.com/…","""Ajaib! Tato Elektronik Buatan …","""""Di masa depan, apa yang kami …"
"""2019-10-30""","""https://www.cnbcindonesia.com/…","""Chairul Tanjung Hadiri BSI GIF…","""Menurutnya, karakter anak-anak…"
"""2022-03-28""","""https://www.cnbcindonesia.com/…","""11 Tanda Rumah Anda Sudah Terc…","""Jakarta, CNBC Indonesia - Polu…"


## 1. Profiling dengan DuckDB `SUMMARIZE`

DuckDB membaca kumpulan file Parquet secara langsung untuk merangkum tipe data, nilai minimum/maksimum, jumlah unik, dan persentase null.

In [3]:
raw_sql_path = Path(RAW_GLOB).as_posix().replace("'", "''")
with duckdb.connect() as connection:
    relation = connection.sql(
        f"SUMMARIZE SELECT * FROM read_parquet('{raw_sql_path}')"
    )
    profil = pl.DataFrame(
        relation.fetchall(),
        schema=[column[0] for column in relation.description],
        orient="row",
    )

profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[38,2]"
"""date""","""VARCHAR""","""2019-05-10""","""NaT""",379,null,null,null,null,null,1149789,0.00
"""link""","""VARCHAR""","""https://20.detik.com/blak-blak…","""https://yogyakarta.kompas.com/…",949847,null,null,null,null,null,1149789,0.00
"""title""","""VARCHAR""","""""100% Manusia Film Festival 20…","""﻿Unsur-unsur Surat Pribadi, Le…",851378,null,null,null,null,null,1149789,0.00
"""text""","""VARCHAR""",""" Okezone Up…","""• Perusahaan Berkomitmen untuk…",771840,null,null,null,null,null,1149789,0.00


## 2. Deteksi Missing Values

Selain null asli, nilai kosong dan literal `NaT` pada kolom tanggal diperlakukan sebagai missing value semantik.

In [4]:
null_counts = raw_lf.select(
    *[pl.col(column).null_count().alias(column) for column in required_columns]
).collect()

null_summary = (
    null_counts
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns(
        (pl.col("jumlah_null") / jumlah_baris * 100).round(2).alias("persen_null")
    )
    .sort("jumlah_null", descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""date""",0,0.0
"""link""",0,0.0
"""title""",0,0.0
"""text""",0,0.0


In [5]:
semantic_missing = raw_lf.select(
    pl.col("date").str.strip_chars().eq("NaT").sum().alias("date_NaT"),
    *[
        pl.col(column).str.strip_chars().eq("").sum().alias(f"{column}_kosong")
        for column in required_columns
    ],
).collect()
semantic_missing

date_NaT,date_kosong,link_kosong,title_kosong,text_kosong
u32,u32,u32,u32,u32
101421,0,0,0,0


## 3. Penanganan Missing Values

Dataset tidak memiliki null asli pada empat kolom utama. Nilai `NaT` tidak diimputasi karena tanggal publikasi tidak dapat ditebak secara bertanggung jawab; baris tersebut dikeluarkan dari analisis temporal.

In [6]:
df_missing_handled = (
    raw_lf
    .select(
        pl.col("date").str.strip_chars().str.to_date("%Y-%m-%d", strict=False).alias("date"),
        pl.col("link").str.strip_chars().alias("link"),
        pl.col("title").str.strip_chars().str.replace_all(r"\s+", " ").alias("title"),
        pl.col("text").str.strip_chars().str.replace_all(r"\s+", " ").alias("text"),
    )
    .filter(
        pl.col("date").is_not_null(),
        pl.col("link").is_not_null() & pl.col("link").str.contains(r"^https?://"),
        pl.col("title").is_not_null() & pl.col("title").ne(""),
        pl.col("text").is_not_null() & pl.col("text").ne(""),
    )
    .with_columns(
        pl.col("title").str.len_chars().alias("title_length"),
        pl.col("text").str.len_chars().alias("text_length"),
    )
)

jumlah_setelah_missing = df_missing_handled.select(pl.len()).collect().item()
print(f"Baris sebelum penanganan missing: {jumlah_baris:,}")
print(f"Baris sesudah penanganan missing: {jumlah_setelah_missing:,}")
print(f"Baris tanggal tidak valid yang dihapus: {jumlah_baris - jumlah_setelah_missing:,}")

Baris sebelum penanganan missing: 1,149,789
Baris sesudah penanganan missing: 1,048,368
Baris tanggal tidak valid yang dihapus: 101,421


## 4. Deteksi dan Penanganan Outlier

Kolom numerik yang relevan adalah panjang judul dan isi artikel. Batas outlier dihitung dengan metode IQR. Teks asli tetap dipertahankan, sedangkan fitur panjang versi bersih di-*winsorize* dengan `clip` agar nilai ekstrem tidak mendominasi analisis numerik.

In [7]:
quantiles = df_missing_handled.select(
    pl.col("title_length").quantile(0.25).alias("title_q1"),
    pl.col("title_length").quantile(0.75).alias("title_q3"),
    pl.col("text_length").quantile(0.25).alias("text_q1"),
    pl.col("text_length").quantile(0.75).alias("text_q3"),
).collect().row(0, named=True)

def hitung_batas_iqr(q1, q3):
    iqr = q3 - q1
    return max(0, q1 - 1.5 * iqr), q3 + 1.5 * iqr

title_lower, title_upper = hitung_batas_iqr(quantiles["title_q1"], quantiles["title_q3"])
text_lower, text_upper = hitung_batas_iqr(quantiles["text_q1"], quantiles["text_q3"])

batas_outlier = pl.DataFrame({
    "kolom": ["title_length", "text_length"],
    "batas_bawah": [title_lower, text_lower],
    "batas_atas": [title_upper, text_upper],
})
batas_outlier

kolom,batas_bawah,batas_atas
str,f64,f64
"""title_length""",27.0,115.0
"""text_length""",0.0,5195.0


In [8]:
df_outlier_handled = df_missing_handled.with_columns(
    ((pl.col("title_length") < title_lower) | (pl.col("title_length") > title_upper)).alias("title_length_outlier"),
    ((pl.col("text_length") < text_lower) | (pl.col("text_length") > text_upper)).alias("text_length_outlier"),
    pl.col("title_length").clip(title_lower, title_upper).alias("title_length_clean"),
    pl.col("text_length").clip(text_lower, text_upper).alias("text_length_clean"),
)

outlier_summary = df_outlier_handled.select(
    pl.col("title_length_outlier").sum().alias("outlier_judul"),
    pl.col("text_length_outlier").sum().alias("outlier_teks"),
).collect()
outlier_summary

outlier_judul,outlier_teks
u32,u32
13235,41766


## 5. Deduplikasi

Kolom `link` digunakan sebagai kunci bisnis karena satu URL merepresentasikan satu artikel berita.

In [9]:
KOLOM_KUNCI = "link"
total_sebelum_dedup = df_outlier_handled.select(pl.len()).collect().item()
baris_unik = df_outlier_handled.select(pl.col(KOLOM_KUNCI).n_unique()).collect().item()

print(f"Total baris sebelum deduplikasi: {total_sebelum_dedup:,}")
print(f"Link unik: {baris_unik:,}")
print(f"Duplikat link: {total_sebelum_dedup - baris_unik:,}")

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI],
    keep="first",
    maintain_order=True,
)
df_dedup.select(pl.len().alias("baris_setelah_deduplikasi")).collect()

Total baris sebelum deduplikasi: 1,048,368
Link unik: 928,082
Duplikat link: 120,286


baris_setelah_deduplikasi
u32
928082


## 6. Validasi Data Dasar

Aturan validasi disesuaikan dengan dataset berita: kolom wajib tidak boleh null, URL harus unik, tanggal harus berada pada rentang dataset, dan fitur panjang hasil winsorization harus berada di dalam batas IQR.

In [10]:
df_bersih = df_dedup.with_columns(
    pl.col("link")
    .str.extract(r"https?://(?:www\.)?([^/]+)", 1)
    .str.to_lowercase()
    .alias("source_domain")
)

validasi_polars = df_bersih.select(
    pl.len().alias("total_baris"),
    pl.sum_horizontal(
        [pl.col(column).is_null().sum() for column in required_columns]
    ).alias("pelanggaran_null"),
    (pl.len() - pl.col("link").n_unique()).alias("pelanggaran_duplikat_link"),
    ((pl.col("title_length_clean") < title_lower) | (pl.col("title_length_clean") > title_upper)).sum().alias("pelanggaran_batas_judul"),
    ((pl.col("text_length_clean") < text_lower) | (pl.col("text_length_clean") > text_upper)).sum().alias("pelanggaran_batas_teks"),
    pl.col("date").min().alias("tanggal_awal"),
    pl.col("date").max().alias("tanggal_akhir"),
).collect()
validasi_polars

total_baris,pelanggaran_null,pelanggaran_duplikat_link,pelanggaran_batas_judul,pelanggaran_batas_teks,tanggal_awal,tanggal_akhir
u32,u32,u32,u32,u32,date,date
928082,0,0,0,0,2019-05-10,2023-10-01


In [11]:
for aturan in [
    "pelanggaran_null",
    "pelanggaran_duplikat_link",
    "pelanggaran_batas_judul",
    "pelanggaran_batas_teks",
]:
    assert validasi_polars.item(0, aturan) == 0, f"Validasi gagal: {aturan}"

assert validasi_polars.item(0, "tanggal_awal").year >= 2019
assert validasi_polars.item(0, "tanggal_akhir").year <= 2023
print("Semua aturan validasi Polars lulus.")

Semua aturan validasi Polars lulus.


## 7. Simpan Data Bersih

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_bersih.sink_parquet(OUTPUT_PATH, compression="zstd", maintain_order=True)

jumlah_bersih = pl.scan_parquet(OUTPUT_PATH).select(pl.len()).collect().item()
print(f"Baris sebelum cleaning: {jumlah_baris:,}")
print(f"Baris sesudah cleaning: {jumlah_bersih:,}")
print(f"Baris yang dihapus: {jumlah_baris - jumlah_bersih:,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")
print(f"Ukuran file: {OUTPUT_PATH.stat().st_size / (1024 ** 2):,.2f} MB")

Baris sebelum cleaning: 1,149,789
Baris sesudah cleaning: 928,082
Baris yang dihapus: 221,707
Data bersih tersimpan di: C:\Users\endo1\Documents\SEMESTER  7\big data\tugas1-MohammadRavlindoS\data\processed\dataset_bersih.parquet
Ukuran file: 691.27 MB


## 8. Verifikasi Hasil dengan DuckDB

DuckDB membaca file hasil akhir untuk memastikan file dapat digunakan kembali tanpa bergantung pada objek Polars di memori.

In [13]:
output_sql_path = OUTPUT_PATH.as_posix().replace("'", "''")
with duckdb.connect() as connection:
    validasi_duckdb = connection.sql(f"""
        SELECT
            COUNT(*) AS total_baris,
            SUM(CASE WHEN date IS NULL OR link IS NULL OR title IS NULL OR text IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
            COUNT(*) - COUNT(DISTINCT link) AS pelanggaran_duplikat_link,
            SUM(CASE WHEN title_length_clean < {title_lower} OR title_length_clean > {title_upper} THEN 1 ELSE 0 END) AS pelanggaran_batas_judul,
            SUM(CASE WHEN text_length_clean < {text_lower} OR text_length_clean > {text_upper} THEN 1 ELSE 0 END) AS pelanggaran_batas_teks
        FROM read_parquet('{output_sql_path}')
    """).pl()

validasi_duckdb

total_baris,pelanggaran_null,pelanggaran_duplikat_link,pelanggaran_batas_judul,pelanggaran_batas_teks
i64,"decimal[38,0]",i64,"decimal[38,0]","decimal[38,0]"
928082,0,0,0,0


In [14]:
with duckdb.connect() as connection:
    profil_bersih = connection.sql(
        f"SUMMARIZE SELECT * FROM read_parquet('{output_sql_path}')"
    ).pl()

profil_bersih

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""date""","""DATE""","""2019-05-10""","""2023-10-01""",524,"""2023-05-04 07:47:19.070686""",null,"""2023-02-16""","""2023-05-21""","""2023-07-29""",928082,0.00
"""link""","""VARCHAR""","""https://20.detik.com/blak-blak…","""https://yogyakarta.kompas.com/…",846088,null,null,null,null,null,928082,0.00
"""title""","""VARCHAR""","""""100% Manusia Film Festival 20…","""﻿Unsur-unsur Surat Pribadi, Le…",768966,null,null,null,null,null,928082,0.00
"""text""","""VARCHAR""",""""" Jakarta (ANTARA) - Ketua ASE…","""• Perusahaan Berkomitmen untuk…",781378,null,null,null,null,null,928082,0.00
"""title_length""","""UINTEGER""","""8""","""199""",205,"""70.83904978223907""","""16.558804999198024""","""60""","""69""","""81""",928082,0.00
"""text_length""","""UINTEGER""","""14""","""70210""",11144,"""2333.246103253807""","""1462.3452310200046""","""1485""","""2133""","""2894""",928082,0.00
"""title_length_outlier""","""BOOLEAN""","""false""","""true""",2,null,null,null,null,null,928082,0.00
"""text_length_outlier""","""BOOLEAN""","""false""","""true""",2,null,null,null,null,null,928082,0.00
"""title_length_clean""","""UINTEGER""","""27""","""115""",91,"""70.6999984915126""","""16.00697287789736""","""60""","""69""","""81""",928082,0.00


## Catatan Milestone 2

Dimensi kualitas yang paling bermasalah adalah validitas tanggal dan duplikasi URL. Sebanyak 101.421 baris memiliki tanggal `NaT` dan dikeluarkan karena tanggal publikasi tidak dapat diimputasi secara bertanggung jawab. Outlier panjang judul dan isi ditangani dengan winsorization pada fitur numeriknya, sementara teks asli tetap dipertahankan. Selanjutnya, 120.286 duplikat dihapus menggunakan `link` sebagai kunci bisnis.

Dataset akhir harus memenuhi aturan berikut: kolom wajib tanpa null, setiap URL unik, tanggal berada pada periode 2019-2023, serta nilai panjang hasil winsorization berada dalam batas IQR. Hasil cleaning disimpan sebagai `data/processed/dataset_bersih.parquet` untuk digunakan pada Milestone 3.